# Setup

In [1]:
#pip install -qU langchain langchain-ollama

In [ ]:
import os
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel

from pydantic import Field
import json
from typing import Literal
os.environ["GOOGLE_API_KEY"] = None #include your own api key here

In [3]:
patientState = {
    "age": 19,
    "nationality": "Egyptian",
    "gender": "male",

    "goals": [
        "Manage academic stress",
        "Improve sleep consistency",
        "Build confidence in social situations"
    ],

    "symptoms": [
        {
            "name": "excessive_worry",
            "severity": "moderate",
            "duration": "6 weeks",
            "description": "Frequent worries about academic performance and deadlines"
        },
        {
            "name": "sleep_difficulty",
            "severity": "mild",
            "duration": "3 weeks",
            "description": "Difficulty falling asleep before exams"
        },
        {
            "name": "social_anxiety",
            "severity": "mild",
            "duration": "2 months",
            "description": "Nervousness when speaking in groups or meeting unfamiliar people"
        }
    ],

    "patientHistory": {
        "previousMentalHealthTreatment": "None reported",
        "medicalHistory": "No relevant medical history reported",
        "medicationHistory": "No current medication reported",
        "familyMentalHealthHistory": "Unknown",
        "majorLifeEvents": [
            "Increased academic workload this semester"
        ],
        "copingStrategies": [
            "Listening to music",
            "Talking to a close friend"
        ],
        "socialSupport": "Has a few supportive friends",
        "riskHistory": "Not yet assessed",
    }
}

# Session Planner Agent

In [4]:
class SessionPlan(BaseModel):
    session_goal: str
    possible_interventions: list[str]
    session_objectives: list[str] = Field(default_factory=list)

In [5]:
treatment_plan = """
Client Name: David Miller
DOB: 10/14/1988
Age: 37
Date: 10/06/2026

Diagnosis: (F32.2) Major Depressive Disorder, Single Episode, Severe without Psychotic Features

Clinical Information Gaps: None identified from the supplied data.

Problem 1: Severe Major Depressive Disorder (MDD) with symptoms impacting daily functioning (e.g., work routine, sleep patterns) following a corporate downsizing event 6 months ago.

Goal: Reduce depressive symptoms from severe to mild range (PHQ-9 score from 19 to <7), re-engage in daily work routine, and restore sleep patterns within 12 weeks.

Objective 1: Establish therapeutic alliance and collaboratively develop a treatment plan based on Behavioral Activation (BA).
    Intervention 1-1: Conduct a comprehensive assessment, including symptom severity (e.g., PHQ-9), functional impact, and patient goals. Rationale: Essential for understanding the patient's presentation and tailoring treatment.
    Intervention 1-2: Psychoeducation about depression, its course, and the rationale for Behavioral Activation (BA). Rationale: Provides patient understanding and buy-in for treatment [G1].
    Intervention 1-3: Collaboratively develop a treatment plan, including specific, measurable goals and the use of BA. Rationale: Empowers the patient and ensures treatment aligns with their needs [G1].

Objective 2: Implement Behavioral Activation (BA) to increase engagement in activities linked to improved mood.
    Intervention 2-1: Identify activities that are linked to improved mood and develop a schedule of activities. Rationale: BA focuses on increasing engagement in rewarding or mood-improving activities to reduce avoidance and improve mood [G2, G4].
    Intervention 2-2: Monitor activity scheduling and mood changes. Rationale: Tracking progress helps identify patterns and adjust the plan as needed [G2, G4].
    Intervention 2-3: Address barriers to activity engagement (e.g., lack of motivation, social withdrawal). Rationale: Helps the patient overcome obstacles to implementing the BA plan [G2, G4].

Objective 3: Monitor progress, adjust treatment as needed, and plan for relapse prevention.
    Intervention 3-1: Regularly monitor symptom severity (e.g., PHQ-9) and functional improvement. Rationale: Tracks progress towards goals and informs treatment adjustments [G1].
    Intervention 3-2: Review treatment response and adjust interventions as needed. Rationale: Ensures treatment remains effective and responsive to the patient's needs [G1].
    Intervention 3-3: Discuss relapse prevention strategies, including identifying triggers and early warning signs. Rationale: Prepares the patient for potential future episodes [G1].

Monitoring and Reassessment:
- Monitor PHQ-9 score at regular intervals (e.g., every 2-4 weeks) to track symptom severity.
- Assess functional improvement related to work routine and sleep patterns weekly.
- Review treatment progress and adjust the plan based on response.
- Schedule a follow-up assessment within 2-4 weeks if symptoms are improving to discuss progress and plan next steps [G3].
"""

In [6]:
previous_session_summary = {
    "Session 1": """
    David discussed how losing his job affected his mood, sleep,
    and daily routine. He agreed to track his mood and activities.
    """,
    "Session 2": """
    David reported low motivation and difficulty maintaining a routine.
    He identified walking and preparing breakfast as manageable activities.
    """,
    "Session 3": """
    David completed mood tracking but only tried walking once.
    He struggled to get started in the mornings and felt discouraged.
    The session explored barriers to activity and setting smaller goals.
    """
}

In [7]:
INTERVENTIONS = {
    'cognitive_rehearsal': 'Mentally practice a response to a situation.',
    'motivational_interviewing': 'Explore motivation and ambivalence (MI-derived).',
    'graded_exposure': 'Gradually approach fears using a clinician-approved plan.',
    'behavioral_activation': 'Explore small, meaningful activities and barriers.',
    'behavioral_experiments': 'Test a prediction collaboratively through observation.',
    'challenging_core_beliefs': 'Explore broad beliefs about oneself.',
    'identifying_cognitive_distortions': 'Recognize patterns of biased thinking.',
    'attributional_analysis': 'Consider different explanations for events.',
    'reframing': 'Explore an alternative way of understanding a situation.',
    'psychoeducation': 'Explain a relevant concept using reviewed material.',
    'socratic_questioning': 'Explore assumptions through collaborative questions.',
    'cognitive_defusion': 'Notice thoughts as thoughts (ACT-derived).',
    'cognitive_restructuring': 'Examine thoughts and alternative perspectives.',
    'decatastrophizing': 'Explore feared outcomes and coping options.',
    'evidence_examination': 'Consider evidence supporting and challenging a belief.',
    'ipt_interpersonal_inventory': 'Map important relationships.',
    'ipt_communication_analysis': 'Review a specific interpersonal exchange.',
    'ipt_role_play': 'Rehearse an interpersonal conversation.',
    'ipt_decision_analysis': 'Explore interpersonal choices.',
    'ipt_encouragement_of_affect': 'Explore emotions in relationships.',
    'ipt_interpersonal_incidents': 'Explore recent interpersonal experiences.',
    'ipt_interpersonal_problem_solving': 'Consider responses to relationship difficulties.',
    'ipt_role_transitions': 'Explore significant changes in life roles.',
    'guided_exploration': 'Collaboratively explore a specific concern to understand feelings, experiences, patterns and possible contributing factors.',
}
INTERVENTIONS["support"] = "Provide supportive conversation, acknowledge concerns, and respect the patient’s pace."

In [8]:
#  Guided exploration can be suggested for assessment/formulation work.

In [9]:
Session_planner_agent = create_agent(
    model=ChatGoogleGenerativeAI(
        model="gemma-4-26b-a4b-it",
        temperature=0.6,
        thinking_level="minimal",
    ),
    tools=[],
    response_format=SessionPlan,
    system_prompt="""
    You are a therapy session planner.
    
    Use the treatment plan and previous session summary to determine
    the main goal of the upcoming therapy session.
    
    Output:
    one Concise specific, realistic session goal.
    Suggest up to 4 suitable interventions from AVAILABLE INTERVENTIONS, guided by the treatment plan.
    Suggestions are optional and may be empty; do not force an intervention.
    Also provide 2-4 clear, ordered session_objectives that serve the session goal.
    Objectives may involve understanding a problem, intervention work, or reviewing progress.
    Keep objectives realistic for one session; do not force a formal intervention.
    """,
)


result = Session_planner_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"""

Treatment Plan:
{treatment_plan}

Patient Information:
{patientState}

Previous Session Summary:
{previous_session_summary}

AVAILABLE INTERVENTIONS:
{INTERVENTIONS}

"""
        }
    ]
})

In [10]:
Session_plan = result["structured_response"]
print(Session_plan.model_dump_json(indent=2))

session_objectives = list(Session_plan.session_objectives) or [Session_plan.session_goal]
planner_interventions = {
    name: INTERVENTIONS[name]
    for name in Session_plan.possible_interventions
    if name in INTERVENTIONS
}

{
  "session_goal": "Address barriers to activity engagement and refine the Behavioral Activation plan to improve morning routine consistency.",
  "possible_interventions": [
    "behavioral_activation",
    "motivational_interviewing",
    "support"
  ],
  "session_objectives": [
    "Review progress with mood and activity tracking from the previous week.",
    "Identify specific psychological or environmental barriers preventing morning routine engagement.",
    "Collaboratively develop a more granular, 'micro-goal' version of the activity schedule.",
    "Assess current motivation levels regarding the upcoming week's activities."
  ]
}


# Decision Agent

In [11]:
Action = Literal["clarify", "explore", "reflect", "validate", "answer", "intervene"]

class TherapyDecision(BaseModel):
    selected_action: Action = Field(description="The conversational move for this turn.")
    goal: str = Field(default="", description=". a concise sentence, saying the Immediate goal of the next response not what the response should say")
    intervention_id: str | None = Field(default=None, description="Planner-listed intervention being used, if any.")
    objective_complete: bool = False
    phase_complete: bool = False


In [12]:
# this is just for now will be updated
av_actions = {
    "Clarify": "Ask questions to resolve ambiguity or missing information.",
    "Explore": "Understand the user's thoughts, feelings, and experiences.",
    "Reflect": "Restate the user's meaning to show understanding.",
    "Validate": "Acknowledge the user's feelings and perspective without judgment.",
    "Answer": "Provide relevant information or respond directly to the user's question.",
    "Intervene": "Apply a suitable therapeutic technique to support the user's goals.",
}

In [13]:
actions_str = "\n".join([f"- {action}: {desc}" for action, desc in av_actions.items()])

In [14]:
reasoning_agent = create_agent(
    model= ChatGoogleGenerativeAI(
        model="gemma-4-26b-a4b-it",
        temperature=0.6,
        thinking_level="minimal")
    ,
    tools=[],
    response_format=TherapyDecision,
    system_prompt= f"""
You are the therapeutic reasoning component. Choose the most useful
next conversational move, not merely the safest or most familiar one.

Available actions:
{actions_str}

Rules:
- Determine what the patient needs from this particular turn.
- Answer direct questions directly. Do not turn every message into
  a request for more information.
- When sufficient information is available, contribute something useful:
  explain a pattern, offer an alternative perspective, suggest a practical
  step, or help the patient make a decision.
- Ask a question only when its answer would materially change the
  response or essential information is missing.
- Never ask for information already present in the conversation.
- Build on previous turns instead of repeatedly exploring the same topic.
- Gently examine harsh self-judgments when appropriate.
- Do not force an intervention into ordinary conversation.
- Take initiative when a useful direction is evident.
- Treat safety concerns seriously.
- Follow session plan when appropriate 

Session Tracking:
- Follow the current phase and objective without forcing an intervention.
- Choose one of the six conversational actions for each turn.
- Use intervene only for a suitable intervention the patient agrees to try.
- Set intervention_id to the planner-listed intervention when it is being used; otherwise leave it null.
- Mark objective_complete=True only when the current objective has actually been completed.
- Mark phase_complete=True only when the current session phase is finished.
- In the final CLOSING phase, phase_complete=True marks the session finished.

Return only: selected_action, goal, intervention_id, objective_complete, phase_complete.

Do not write the patient-facing response. Keep guidance concise.
"""
)

# Therapist Agent

In [15]:
therapist_agent = create_agent(
    model=ChatGoogleGenerativeAI(
        model="gemma-4-26b-a4b-it",
        temperature=0.6,
        thinking_level="minimal"),
    tools=[],
    system_prompt="""
    You are the conversational therapist component of a larger mental health agentic system.
    You are being Guided by a therapy decision agent.

    Response Rules:
    - Respond in a natural, empathetic, emotionally aware, and respectful way.
    - Generally respond in a clear, and simple way.
    - Allow appropriate humor and playful responses when they fit the patient's tone and the conversation context.
    - Minimize repetitive openings like "that sounds like" or "I am sorry to hear" and vary responses naturally rather than following a fixed pattern
    - Do not default to acknowledging feelings followed by a question. Respond directly, or ask a question depending on what the patient needs.
    - Avoid repeating the same validation, question, or suggestion across turns. Use the recent dialogue to build on answers already given.
    - Do not assume the patient is feeling bad or experiencing distress without evidence.
    - Ask at most one question, only when useful.
    - Prefer 1-2 short sentences, usually under 30 words; use more only if  necessary.
    - Try to push the conversation forward, don't merely paraphrase what the patient just said.

    Session Rules:
    - Focus on what the patient said while also following the current phase, and objectives.
    - When a patient mentions suicidal thoughts, respond to their concerns with support and care and continue therapy. 
    - Just a mention of suicide or patient being suicidal does not mean its an emergency. 
    
    Output:
    A concise response to the user.
"""
)

# sim


In [16]:
def get_message_text(content):
    """Extract text from a model response."""
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        return "\n".join(
            block["text"]
            for block in content
            if isinstance(block, dict)
            and block.get("type") == "text"
        )

    return str(content)

In [23]:
PHASES = ["CHECK_IN_REVIEW", "AGENDA", "APPLY_AGENDA", "EVALUATE", "HOMEWORK", "CLOSING"]
PHASE_GOALS = {
    "CHECK_IN_REVIEW": "Check how the patient is doing and review relevant recent progress.",
    "AGENDA": "Collaboratively agree on the session focus.",
    "APPLY_AGENDA": "Work on the current planned objective, with optional agreed interventions.",
    "HOMEWORK": "Discuss an optional next step or homework.",
    "CLOSING": "Summarize and bring the session to a close."
}

session_state = {
    "current_phase": PHASES[0],
    "status": "ACTIVE",
    "objective_index": 0,
    "active_intervention": None,
    "intervention_history": [],
    "objective_results": {},
}
sessionHistory = "No previous turns."

def current_objective():
    i = session_state["objective_index"]
    return session_objectives[i] if i < len(session_objectives) else None

def finish_intervention(outcome):
    active = session_state["active_intervention"]
    if active:
        session_state["intervention_history"].append({"intervention": active, "outcome": outcome})
        session_state["active_intervention"] = None

def advance_phase():
    if session_state["current_phase"] == "CLOSING":
        session_state["status"] = "ENDED"
    else:
        index = PHASES.index(session_state["current_phase"])
        session_state["current_phase"] = PHASES[index + 1]

def update_session(decision):
    """Track objectives/interventions; ONLY phase_complete advances the phase."""
    phase = session_state["current_phase"]

    if phase == "APPLY_AGENDA" and decision.selected_action == "intervene":
        intervention = decision.intervention_id
        if intervention in planner_interventions:
            if session_state["active_intervention"] != intervention:
                finish_intervention("CHANGED")
                session_state["active_intervention"] = intervention

    if phase == "APPLY_AGENDA" and decision.objective_complete and current_objective():
        session_state["objective_results"][session_state["objective_index"]] = "COMPLETED"
        session_state["objective_index"] += 1
        finish_intervention("COMPLETED")

    if decision.phase_complete:
        if phase == "APPLY_AGENDA":
            finish_intervention("ENDED_WITH_PHASE")
        advance_phase()


In [24]:
while session_state["status"] == "ACTIVE":
    patientQuery = input("\nPatient: ").strip()
    if patientQuery.lower() == "exit":
        print("Simulation stopped by user.")
        break
    if not patientQuery:
        continue

    recentHistory = sessionHistory[-12000:]
    phase = session_state["current_phase"]
    reasoning_result = reasoning_agent.invoke({"messages": [{
        "role": "user",
        "content": f"""
PATIENT STATE: {json.dumps(patientState, ensure_ascii=False, default=str)}
SESSION PLAN: {Session_plan.model_dump_json()}
CURRENT PHASE: {phase} — {PHASE_GOALS[phase]}
CURRENT OBJECTIVE: {current_objective() if phase == 'APPLY_AGENDA' else None}
APPROVED INTERVENTIONS: {json.dumps(planner_interventions)}
ACTIVE INTERVENTION: {session_state['active_intervention']}
INTERVENTION HISTORY: {json.dumps(session_state['intervention_history'])}
RECENT HISTORY: {recentHistory}
PATIENT: {patientQuery}
"""
    }]})
    decision = reasoning_result["structured_response"]
    print("\n--- Therapy Decision ---")
    print(decision.model_dump_json(indent=2))

    update_session(decision)
    active = session_state["active_intervention"]
    phase = session_state["current_phase"]
    therapist_result = therapist_agent.invoke({"messages": [{
        "role": "user",
        "content": f"""
PATIENT STATE: {json.dumps(patientState, ensure_ascii=False, default=str)}
PATIENT QUERY: {patientQuery}
RECENT SESSION HISTORY: {recentHistory}
SESSION PLAN: {Session_plan.model_dump_json()}
CURRENT PHASE: {phase} — {PHASE_GOALS[phase]}
CURRENT OBJECTIVE: {current_objective() if phase == 'APPLY_AGENDA' else None}
THERAPY REASONING AGENT OUTPUT: {decision.model_dump_json()}
INTERVENTION GUIDANCE: {planner_interventions.get(active) if active else None}
"""
    }]})
    therapist_response = get_message_text(therapist_result["messages"][-1].content).strip()
    print("\n--- Therapist ---")
    print(therapist_response)
    sessionHistory += f"\nPatient: {patientQuery}\nTherapist: {therapist_response}\n"
    print("\n--- Session State ---")
    print(json.dumps(session_state, indent=2))


Patient:  hey



--- Therapy Decision ---
{
  "selected_action": "answer",
  "goal": "Acknowledge the user's greeting and initiate the session by transitioning to the session goal of reviewing progress.",
  "intervention_id": null,
  "objective_complete": false,
  "phase_complete": false
}

--- Therapist ---
Hey there! How have you been feeling since we last connected?

--- Session State ---
{
  "current_phase": "CHECK_IN_REVIEW",
  "status": "ACTIVE",
  "objective_index": 0,
  "active_intervention": null,
  "intervention_history": [],
  "objective_results": {}
}



Patient:  Ive been feeling depressed



--- Therapy Decision ---
{
  "selected_action": "explore",
  "goal": "Understand the depth and nature of the patient's depressive symptoms to better assess the clinical state.",
  "intervention_id": null,
  "objective_complete": false,
  "phase_complete": false
}

--- Therapist ---
I'm sorry you're going through that. Can you tell me a bit more about what that depression feels like for you lately?

--- Session State ---
{
  "current_phase": "CHECK_IN_REVIEW",
  "status": "ACTIVE",
  "objective_index": 0,
  "active_intervention": null,
  "intervention_history": [],
  "objective_results": {}
}



Patient:  Just horrible sleep, bad grades, and always tired


...


--- Therapy Decision ---
{
  "selected_action": "explore",
  "goal": "Assess the patient's readiness to move on to the session objectives or if they have specific topics they want to address.",
  "intervention_id": null,
  "objective_complete": false,
  "phase_complete": false
}

--- Therapist ---
My mistake again! Let's stay right here in the moment then. Since we're still fresh in our conversation, is there anything specific on your mind right now?

--- Session State ---
{
  "current_phase": "CHECK_IN_REVIEW",
  "status": "ACTIVE",
  "objective_index": 0,
  "active_intervention": null,
  "intervention_history": [],
  "objective_results": {}
}



Patient:  like can we move on to agenda and interventions



--- Therapy Decision ---
{
  "selected_action": "answer",
  "goal": "Acknowledge the patient's request to move to the agenda and interventions and transition to the session plan.",
  "intervention_id": null,
  "objective_complete": false,
  "phase_complete": false
}

--- Therapist ---
Of course, let's get right to it. Since we've talked about your sleep, I'd like to focus our agenda on looking at your daily routines and finding ways to help you feel more energized. 

Does that sound like a good starting point for you?

--- Session State ---
{
  "current_phase": "CHECK_IN_REVIEW",
  "status": "ACTIVE",
  "objective_index": 0,
  "active_intervention": null,
  "intervention_history": [],
  "objective_results": {}
}



Patient:  exit


Simulation stopped by user.
